# Laya V2.8.1 — RoPE/SDPA-corrected fast Window + Global Memory (Layer 15)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V28_Layer15_FastWindowMemory_Colab.ipynb)

**V2.8.1 fixes:** preserve ModernBERT SDPA mask/position kwargs, apply the pretrained per-layer RoPE to Q/K, test teacher standalone replay before training, correct centered-window coverage, use disjoint end-to-end audit decisions, and checkpoint immediately after each 400-step run. The more balanced routing initialization is an experiment, not a validated improvement. Restart the Colab runtime and run all cells from the top.

**Research target, not a result:** validation cosine ≥ **0.99 after at most 400 successful parameter updates**, on **held-out teacher attention I/O**. No guarantee that the target is attainable.

Based on V2.6 Local32 (reported 0.8816 at step 400 and about 0.8972 at step 1200 of refinement). Instead of training the same recurrent architecture for longer, V2.8 tests a fundamentally faster **PDelta3-inspired** replacement:

- Exact **bounded bidirectional local attention** with windows **96, 160, 256**; the full-sequence dense attention kernel is *never called by the replacement*.
- A *query-dependent O(T·M) pooled landmark path*, plus **O(T·F·d) bidirectional linear global memory**; both reach distant tokens without dense T×T attention.
- Native pretrained ModernBERT **QKV/RoPE/Wo initialization**, dynamic per-head route fusion, learned local temperature, a low-weight convolutional residual, and optional low-LR Wo calibration.
- Fast path **omits PDelta3's sequential recurrent scan**. This is a new derivative, **not a claim that the exact PDelta3 architecture was preserved**.
- Fixed 400-step local distillation budget **per candidate**, disjoint train/probe/audit examples; audit set is not used to choose the winner.
- Prints local cosine/NMSE at steps 0, 1, 50, ..., 400, validation decision agreement/JS, CUDA latencies, and **fraction of samples entirely covered by each local window**. If most examples are shorter than the window, 0.99 may be misleading as a demonstration of truly subquadratic long-context behavior.

**Important definition:** "without full attention" means **no unbounded full-sequence attention operator at inference**. The local SDPA is still genuine (bounded) attention, so this is a *hybrid* architecture, not an attention-free CeNN. At lengths shorter than W, its local receptive field necessarily contains the whole valid sequence; those examples are explicitly quantified. Distillation uses full attention only inside the unchanged frozen teacher.

**Validity policy:** Select the winner on the fixed probe split, then report a separate untouched audit split and end-to-end agreement. A score ≥0.99 *only on probes* does not meet the audit goal; no test set is used.

In [ ]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Model and V2.8 experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2028
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15
TARGET_LAYERS = (LAYER,)
WINDOWS = (96, 160, 256) #@param {type:"raw"}
LANDMARK_STRIDE = 16 #@param {type:"integer"}
FEATURE_DIM = 96 #@param {type:"integer"}
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 64 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 12 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 16 #@param {type:"integer"}
CHECK_EVERY = 50
TARGET_COSINE = 0.99
OUTPUT_DIR = Path("/content/Laya_V28_Layer15")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert STEPS == 400, "V2.8 comparison is defined at exactly 400 optimizer steps"
assert len(WINDOWS) > 0 and all(isinstance(w, int) and w >= 16 for w in WINDOWS)
assert LANDMARK_STRIDE > 0 and FEATURE_DIM > 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
print("Layer:", LAYER, "| windows:", WINDOWS, "| steps per candidate:", STEPS)
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(entry["x"], **entry["call_kwargs"])
            require_finite(pred, "V2.8 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. V2.8: bounded LocalW + landmarks + global linear memory; no quadratic global attention
class PDeltaV28FastWindowMemory(PDelta3GDN2CLVRAttention):
    """PDelta3-inspired replacement without sequential GDN scan.

    Complexity: local attention O(T*W*d), landmark cross attention
    O(T*ceil(T/stride)*d), and kernel memory O(T*F*d).
    The landmark branch becomes quadratic if stride is fixed while T -> infinity.
    For provably bounded linear scaling, cap its count with adaptive pooling.
    """
    architecture = "pdelta_v28_fast_window_memory"

    def __init__(self, original, *, local_window=96, landmark_stride=16,
                 feature_dim=96, max_landmarks=32):
        super().__init__(
            original, feature_dim=feature_dim, conv_kernel=4,
            chunk_size=32, local_kernel=5,
            local_window=local_window, local_gate_init=0.90,
        )
        self.landmark_stride = int(landmark_stride)
        self.max_landmarks = int(max_landmarks)
        assert self.landmark_stride > 0 and self.max_landmarks > 0
        # Remove obsolete recurrent, causal-convolution and four-way fusion parameters.
        # The inherited methods _local_attention and _global_linear remain in use.
        for name in (
            "wq", "wk", "decay_w", "dt_bias", "erase_w", "erase_b",
            "write_w", "write_b", "route_proj", "route_gate_w",
            "route_gate_b", "q_conv", "k_conv", "v_conv",
            "mix_logits", "mix_gate_w", "local_gate_w",
            "local_gate_b", "log_gain",
        ):
            delattr(self, name)
        # 4.57.6 passes position_ids to ModernBERT, not position_embeddings.
        # Copy the exact pretrained per-layer RoPE instead of silently skipping it.
        self.rotary_emb = copy.deepcopy(original.rotary_emb)
        self.rotary_emb.requires_grad_(False)
        self.layer_idx = int(getattr(original, "layer_id", LAYER))
        h, d = self.num_heads, self.head_dim
        # [local, compressed landmarks, global linear, direct value].
        # Layer 15 is global: do not initialize with ~90% exclusively local routing.
        self.route_logits = nn.Parameter(
            torch.tensor([1.0, 0.2, 0.2, -2.2]).repeat(h, 1)
        )
        self.route_query = nn.Parameter(torch.zeros(h, 4, d))
        self.local_log_temperature = nn.Parameter(torch.zeros(h))
        self.landmark_log_temperature = nn.Parameter(torch.zeros(h))
        self.conv_gain = nn.Parameter(torch.full((h,), -4.0))
        # Pretrained QKV and Wo are copied from the teacher by BaseLayaReplacementAttention.
        # Freeze QKV and allow a very small Wo calibration.
        for p in self.Wqkv.parameters():
            p.requires_grad_(False)
        for p in self.Wo.parameters():
            p.requires_grad_(True)

    def _landmarks(self, q, k, v, valid):
        b, h, t, d = k.shape
        # Choose at most max_landmarks pooled key/value blocks. This ensures
        # O(T*M) with M <= max_landmarks at all sequence lengths.
        stride = max(self.landmark_stride, math.ceil(t / self.max_landmarks))
        blocks = math.ceil(t / stride)
        pad = blocks * stride - t
        mask = valid[:, None, :, None].to(k.dtype)
        packed_valid = F.pad(valid, (0, pad), value=False).reshape(b, blocks, stride)
        counts = packed_valid.sum(-1)
        divisor = counts.clamp_min(1).to(k.dtype)[:, None, :, None]
        kp = F.pad(k * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        vp = F.pad(v * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        pooled_k = kp.sum(3) / divisor
        pooled_v = vp.sum(3) / divisor
        allowed = (counts > 0)[:, None, None, :]
        temperature = self.landmark_log_temperature.clamp(-0.4, 0.4).exp()
        q_scaled = q * temperature[None, :, None, None]
        # Only T x M attention scores; never a dense T x T matrix.
        return F.scaled_dot_product_attention(
            q_scaled, pooled_k, pooled_v, attn_mask=allowed, dropout_p=0.0
        ) * mask

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, position_ids=None, sliding_window_mask=None,
                **kwargs):
        # Exact teacher QKV + RoPE path; apply rotary embeddings BEFORE attention.
        b, t, _ = hidden_states.shape
        qkv = self.Wqkv(hidden_states).reshape(b, t, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        if position_embeddings is None:
            cos, sin = self.rotary_emb(qkv, position_ids=position_ids)
        else:
            cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        valid = _valid_tokens(attention_mask, hidden_states)
        mask = valid[:, None, :, None].to(v.dtype)
        q, k, v = q * mask, k * mask, v * mask
        temperature = self.local_log_temperature.clamp(-0.4, 0.4).exp()
        qlocal = q * temperature[None, :, None, None]
        local = self._local_attention(qlocal, k, v, valid)
        landmarks = self._landmarks(q, k, v, valid)
        global_linear = self._global_linear(q, k, v, mask) * mask
        qnorm = F.normalize(q, dim=-1)
        dynamic = torch.einsum("bhtd,hkd->bhtk", qnorm, self.route_query)
        mixture = torch.softmax(
            self.route_logits.to(q.dtype)[None, :, None, :] + dynamic,
            dim=-1
        )
        fused = (
            mixture[..., 0:1] * local
            + mixture[..., 1:2] * landmarks
            + mixture[..., 2:3] * global_linear
            + mixture[..., 3:4] * v
        )
        local_conv = _depthwise_sequence_conv(v, self.local_weight, causal=False)
        conv_weight = torch.sigmoid(self.conv_gain).to(v.dtype)[None, :, None, None]
        fused = (fused + conv_weight * (local_conv - v)) * mask
        return self.finish(fused, hidden_states)

    def config_dict(self):
        return dict(
            architecture=self.architecture, layer_idx=self.layer_idx,
            local_window=self.local_window, landmark_stride=self.landmark_stride,
            max_landmarks=self.max_landmarks, feature_dim=self.feature_dim,
            global_linear_memory=True, local_windowed_attention=True,
            dense_global_full_attention=False, recurrent_scan=False,
            rope_applied=True, upstream_sdpa_kwargs=True,
        )

def build_v28(window):
    return PDeltaV28FastWindowMemory(
        teacher.encoder.layers[LAYER].attn,
        local_window=int(window),
        landmark_stride=LANDMARK_STRIDE,
        max_landmarks=32,
        feature_dim=FEATURE_DIM,
    ).to(device)

def state_cpu(module):
    return {k: p.detach().cpu().clone() for k, p in module.state_dict().items()}

print("V2.8 architecture ready (no GDN scan; bounded local + 32 capped landmarks)")


In [ ]:
#@title 5. Preflight: real ModernBERT attention signature, finite gradients and long-sequence coverage
preflight_batch = batch(dev_items, 0, min(BATCH, 2))
preflight = attn_io(teacher, LAYER, preflight_batch)
valid = preflight_batch["attention_mask"].bool()
# Validate the exact SDPA call before spending 400 optimization steps.
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
    replay_y, _ = teacher.encoder.layers[LAYER].attn(
        preflight["x"], **preflight["call_kwargs"]
    )
torch.testing.assert_close(replay_y.float(), preflight["y"].float(), atol=1e-3, rtol=1e-3)
print("Original ModernBERT standalone SDPA replay: PASS (all kwargs)")
for w in WINDOWS:
    model = build_v28(w)
    model.train()
    model.out_drop.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        output, _ = model(preflight["x"], **preflight["call_kwargs"])
        active = valid[:, :, None].float()
        loss = (((output.float() - preflight["y"].float()) ** 2) * active).sum() / (
            active.sum() * output.shape[-1]
        ).clamp_min(1.0)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        raw_qkv = model.Wqkv(preflight["x"]).reshape(
            *preflight["x"].shape[:2], 3, model.num_heads, model.head_dim
        )
        rq, rk, _ = raw_qkv.transpose(3, 1).unbind(2)
        c, s = model.rotary_emb(raw_qkv, position_ids=preflight["call_kwargs"]["position_ids"])
        rq, rk = apply_rotary_pos_emb(rq, rk, c, s)
        assert torch.isfinite(rq).all() and torch.isfinite(rk).all(), "non-finite RoPE Q/K"
    require_finite(output, f"V28 W{w} preflight output")
    require_finite(loss, f"V28 W{w} preflight loss")
    parameters = [p for p in model.parameters() if p.requires_grad]
    grads = torch.autograd.grad(loss, parameters, allow_unused=True)
    finite = [g for g in grads if g is not None and torch.isfinite(g).all()]
    if not finite or sum(float(g.abs().sum()) for g in finite) == 0:
        raise RuntimeError(f"W{w}: missing finite nonzero gradients")
    print(f"W{w}: preflight PASS | loss={float(loss):.5f} | trainable={sum(p.numel() for p in parameters):,}")
    del model, output, loss, grads
    torch.cuda.empty_cache()
del preflight_batch, preflight


In [ ]:
#@title 6. Fast 400-step teacher-I/O distillation and checkpoint selection
def score_local(m):
    return m["nmse"] + 1.50 * (1.0 - m["cosine"])

def optimizer_v28(model):
    core, output = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (output if name.startswith("Wo.") else core).append(p)
    params = core + output
    opt = torch.optim.AdamW([
        {"params": core, "lr": 0.003, "base_lr": 0.003},
        {"params": output, "lr": 0.00008, "base_lr": 0.00008},
    ], betas=(0.9, 0.95), weight_decay=2e-4)
    return opt, params

def train_400(model, train_cache, probe_cache, window):
    optimizer, params = optimizer_v28(model)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_score = score_local(best_metrics)
    best_step = 0
    best_state = state_cpu(model)
    history = [{"step": 0, **best_metrics}]
    successes, attempts = 0, 0
    print(f"W{window} step    0/{STEPS}: NMSE={best_metrics['nmse']:.5f} cosine={best_metrics['cosine']:.5f}")

    while successes < STEPS:
        if attempts > STEPS + 300:
            raise RuntimeError("Too many AMP overflow/skipped optimizer steps")
        # A moderate cosine decay + short warmup is more stable than PDelta's
        # former 1e-2 opening learning rate, especially with trainable temperature.
        warm = min(1.0, (successes + 1) / 15.0)
        decay = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * successes / STEPS))
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * warm * decay

        model.train()
        model.out_drop.eval()
        entry = train_cache[attempts % len(train_cache)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            output, _ = model(entry["x"], **entry["call_kwargs"])
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            pred = output.float()
            mse = (((pred - target) ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = ((target ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(pred[entry["valid"]], target[entry["valid"]], dim=-1).mean()
            loss = nmse + 1.50 * (1.0 - cos)
        require_finite(loss, f"V28 W{window} training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        successes += 1
        if successes == 1 or successes % CHECK_EVERY == 0 or successes == STEPS:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": successes, **probe})
            s = score_local(probe)
            if s < best_score:
                best_score, best_metrics, best_step = s, probe, successes
                best_state = state_cpu(model)
            print(
                f"W{window} step {successes:4d}/{STEPS}: "
                f"NMSE={probe['nmse']:.5f} cosine={probe['cosine']:.5f} "
                f"| best cosine={best_metrics['cosine']:.5f} @ {best_step}"
            )
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {
        "best_step": best_step, "best_probe": best_metrics,
        "attempts": attempts, "successful_updates": successes,
        "history": history,
    }


In [ ]:
#@title 7. End-to-end quality and CUDA latency diagnostics
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []
    for off in range(0, limit, BATCH):
        chosen = items[off:min(off + BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "V28 validation logits")
        allowed = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~allowed, -1e9), -1)
        sp = torch.softmax(sl.float().masked_fill(~allowed, -1e9), -1)
        gt = torch.tensor([row["gold_index"] for row in chosen], device=device)
        agree += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gt).sum())
        student_gold += int((sp.argmax(-1) == gt).sum())
        mid = 0.5 * (tp + sp)
        q = 0.5 * (tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        q += 0.5 * (sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(q.cpu().tolist())
        total += len(chosen)
    return dict(
        decisions=total, agreement=agree / max(total, 1),
        mean_js=float(np.mean(js)),
        teacher_gold_accuracy=teacher_gold / max(total, 1),
        student_gold_accuracy=student_gold / max(total, 1),
    )

@torch.inference_mode()
def timed_attention(module, entry, warmup=5, repeats=15):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], **entry["call_kwargs"])
        torch.cuda.synchronize()
        ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], **entry["call_kwargs"])
            torch.cuda.synchronize()
            ms.append((time.perf_counter() - start) * 1000)
    return float(np.median(ms))

def attention_coverage(cache, window):
    lengths = torch.cat([item["valid"].sum(-1).cpu() for item in cache]).numpy()
    # A centered W=96 attends only 47 tokens left and 48 right.
    # Length <= W alone does NOT guarantee all-to-all attention.
    full_context_limit = min((window - 1) // 2, window // 2) + 1
    return {
        "sequences": int(len(lengths)),
        "median_valid_tokens": float(np.median(lengths)),
        "max_valid_tokens": int(np.max(lengths)),
        "fraction_fully_covered_by_local_window": float(np.mean(lengths <= full_context_limit)),
        "full_context_length_limit": int(full_context_limit),
        "sequences_longer_than_window": int(np.sum(lengths > window)),
        "sequences_at_least_2x_window": int(np.sum(lengths >= 2 * window)),
    }

def evaluate_v28(model, audit_cache):
    original = student_eval.encoder.layers[LAYER].attn
    student_eval.encoder.layers[LAYER].attn = model
    try:
        model.eval()
        audit = local_metrics_from_cache(model, audit_cache)
        # Report decision quality only on samples excluded from probe selection.
        quality = validate_pair(student_eval, items=dev_items[PROBE_CACHE_BATCHES * BATCH:])
        teacher_ms = timed_attention(teacher.encoder.layers[LAYER].attn, audit_cache[0])
        v28_ms = timed_attention(model, audit_cache[0])
        return dict(
            audit=audit, e2e=quality, teacher_attention_ms=teacher_ms,
            v28_attention_ms=v28_ms, attention_speedup=teacher_ms / max(v28_ms, 1e-8),
            local_coverage=attention_coverage(audit_cache, model.local_window),
        )
    finally:
        student_eval.encoder.layers[LAYER].attn = original
        student_eval.eval().requires_grad_(False)


In [ ]:
#@title 8. Train W96/W160/W256 for 400 steps each; checkpoint before audit
train_cache = cache_teacher_io(LAYER, train_items, TRAIN_CACHE_BATCHES, offset=LAYER * 13)
probe_cache = cache_teacher_io(LAYER, dev_items, PROBE_CACHE_BATCHES, offset=0)
audit_cache = cache_teacher_io(
    LAYER, dev_items, AUDIT_CACHE_BATCHES, offset=PROBE_CACHE_BATCHES * BATCH
)
results = []
for idx, window in enumerate(WINDOWS):
    seed = SEED + idx * 101
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print("\n" + "=" * 90)
    print(f"V2.8 LAYER {LAYER} — LOCAL {window}, LANDMARKS <=32")
    print("=" * 90)
    model = build_v28(window)
    training = train_400(model, train_cache, probe_cache, window)
    checkpoint = OUTPUT_DIR / f"v28_layer{LAYER}_window{window}.pt"
    payload = {
        "version": "2.8.1", "layer": LAYER, "source_model": SOURCE_MODEL,
        "config": model.config_dict(), "state_dict": state_cpu(model),
        "training": training,
    }
    torch.save(payload, checkpoint)  # durable checkpoint before potentially expensive E2E audit
    print(f"Saved W{window} trained weights: {checkpoint}")
    final = evaluate_v28(model, audit_cache)
    payload["evaluation"] = final
    torch.save(payload, checkpoint)
    row = dict(
        window=window, train=training, evaluation=final,
        checkpoint=str(checkpoint), selection_score=score_local(training["best_probe"]),
    )
    results.append(row)
    p, a, e = training["best_probe"], final["audit"], final["e2e"]
    print(f"PROBE cos={p['cosine']:.5f} NMSE={p['nmse']:.5f} | "
          f"AUDIT cos={a['cosine']:.5f} NMSE={a['nmse']:.5f}")
    print(f"DECISION agreement={e['agreement']:.4f} JS={e['mean_js']:.6f} | "
          f"layer speedup={final['attention_speedup']:.3f}x")
    print(f"COVERAGE: {final['local_coverage']}")
    del model
    torch.cuda.empty_cache()

# ONLY probe selects the winner. The separate audit split is descriptive.
results.sort(key=lambda r: (
    r["selection_score"],
    -r["train"]["best_probe"]["cosine"],
))
winner = results[0]
print(f"\nSELECTED BY PROBE: W{winner['window']} at optimizer step {winner['train']['best_step']}")


In [ ]:
#@title 9. Auditable final report, honest ≥0.99 target gate, and plots
print("\nV2.8 LAYER 15 — CANDIDATE COMPARISON")
print(f"{'W':>5} {'probe_cos':>12} {'audit_cos':>12} {'audit_nmse':>12} {'agreement':>11} {'JS':>10} {'speed':>9} {'full_covered':>14}")
for row in results:
    e = row["evaluation"]
    print(
        f"{row['window']:>5} "
        f"{row['train']['best_probe']['cosine']:>12.5f} "
        f"{e['audit']['cosine']:>12.5f} "
        f"{e['audit']['nmse']:>12.5f} "
        f"{e['e2e']['agreement']:>11.4f} "
        f"{e['e2e']['mean_js']:>10.6f} "
        f"{e['attention_speedup']:>8.3f}x "
        f"{e['local_coverage']['fraction_fully_covered_by_local_window']:>13.1%}"
    )

best_eval = winner["evaluation"]
goal = best_eval["audit"]["cosine"] >= TARGET_COSINE
print("\n400-step cosine target on separate held-out AUDIT:", "PASS" if goal else "NOT REACHED")
print("Winner:", winner["checkpoint"])
if best_eval["local_coverage"]["sequences_longer_than_window"] == 0:
    print("WARNING: audit contains no sequences longer than the winner's local window.")
    print("Do NOT claim this validates a non-full-attention long-context replacement.")
if best_eval["local_coverage"]["fraction_fully_covered_by_local_window"] > 0.5:
    print("WARNING: most audit sequences fit inside the local window; high cosine may be trivial.")

report = {
    "version": "2.8", "source_model": SOURCE_MODEL,
    "target_layer": LAYER, "windows": list(WINDOWS),
    "target_cosine": TARGET_COSINE, "target_steps": STEPS,
    "selected_window_by_probe": winner["window"],
    "audit_target_pass": bool(goal), "official_test_used": False,
    "teacher_only_uses_full_attention": True,
    "replacement_uses_global_dense_attention": False,
    "replace_attention_is_windowed_hybrid": True,
    "selection_rule": "lowest probe NMSE + 1.5*(1-cosine), no audit used for selection",
    "results": results,
}
report_path = OUTPUT_DIR / "v28_layer15_report.json"
report_path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
print("Saved:", report_path)

import matplotlib.pyplot as plt
for row in results:
    h = row["train"]["history"]
    plt.plot([x["step"] for x in h], [x["cosine"] for x in h], marker=".", label=f"W{row['window']}")
plt.axhline(TARGET_COSINE, linestyle="--", color="gray", label="0.99 target")
plt.xlabel("Successful optimizer updates")
plt.ylabel("Probe output cosine")
plt.title("V2.8 layer 15: cosine versus steps")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


In [ ]:
#@title 10. Optional: reload selected V2.8 checkpoint for later inference
# Note: rerun cells 2-4 before this cell to define class, model and teacher source.
# The *student* model uses V2.8 only at layer 15; other attention blocks stay native.
if False:  # Change to True to load the saved experiment winner.
    checkpoint = torch.load(winner["checkpoint"], map_location="cpu", weights_only=False)
    cfg = checkpoint["config"]
    student = build_v28(cfg["local_window"])
    student.load_state_dict(checkpoint["state_dict"], strict=True)
    student_eval.encoder.layers[LAYER].attn = student.eval()
    print("V2.8 installed only at layer", LAYER)
